In [ ]:
# NOTEBOOK 04 - Integración de contaminantes con metadatos de estaciones

"""
en este notebook unifico los dataframes de contaminantes (procesados en el Notebook 02) 
con la información de estaciones (limpiada en el Notebook 03)

para ello:
    extraigo y normalizo el código nacional de estación (NatCode) en todos los contaminantes
    normalizo y depuro NatCode en el dataframe de estaciones, generando df_stations_unique
    filtro los contaminantes para asegurar correspondencia con estaciones válidas
    realizo el merge final contaminante-estación, garantizando integridad many_to_one
    construyo un dataframe integrado (df_all) con todos los contaminantes para el Notebook 05

este notebook deja preparado el dataset unificado necesario para el análisis exploratorio posterior
"""

In [1]:
import pandas as pd
import numpy as np
import os
import re # se usa par abuscar patrones dentro de cadenas de texto

pd.set_option("display.max_columns", None)

# cargo los df de los contaminantes ya limpiados en el notebook 02

contaminantes = {
    "pm25": pd.read_parquet("../data/pollutants/processed_small/pm25_daily.parquet"),
    "pm10": pd.read_parquet("../data/pollutants/processed_small/pm10_daily.parquet"),
    "no2":  pd.read_parquet("../data/pollutants/processed_small/no2_daily.parquet"),
    "o3":   pd.read_parquet("../data/pollutants/processed_small/o3_daily.parquet"),
    "so2":  pd.read_parquet("../data/pollutants/processed_small/so2_daily.parquet"),
    "co":   pd.read_parquet("../data/pollutants/processed_small/co_daily.parquet"),
    "c6h6": pd.read_parquet("../data/pollutants/processed_small/c6h6_daily.parquet")
} # uso un diccionario para no repetir código, si cargo cada contaminante en una variable distinta
# luego tengo que repetir: extraer NatCode, convertir fechas..., además de otras muchas ventajas que supone


# cargo los df limpios del datos para contextualizar del notebook 03
df_hospi = pd.read_csv("../data/context_clean/hospitalizations_clean.csv")
df_mortality = pd.read_csv("../data/context_clean/mortality_clean.csv")
df_population = pd.read_csv("../data/context_clean/population_clean.csv")
df_stations = pd.read_csv("../data/context_clean/stations_clean.csv")



In [2]:
# ahora voy a crear la función extract_natcode(), que es fundamental porque los contaminantes vienen con
# un código largo en Samplingpoint y las estaciones vienen con un código limpio en NatCode y para hacer
# el merge necesitamos que ambos tengan la misma clave


def extract_natcode(samplingpoint):
    """
    extrae el código nacional de estación (8 dígitos) desde la columna Samplingpoint.
    Ejemplo: 'ES1234A_08001001_1_0' -> '08001001'
    """
    match = re.search(r"(\d{8})", str(samplingpoint))
    return match.group(1) if match else None


In [3]:
# ahora aplico la función a todos los contaminantes

for name, df in contaminantes.items():
    # extraigo el NatCode desde Samplingpoint
    df["NatCode"] = df["Samplingpoint"].apply(extract_natcode)
    # aseguro que la columna date es datetime
    df["date"] = pd.to_datetime(df["date"])
    # actualizo el diccionario con el df modificado
    contaminantes[name] = df


In [4]:
# voy a ver si los códigos se extrajeron bien

for name, df in contaminantes.items():
    print(name.upper())
    display(df[["Samplingpoint", "NatCode"]].head(10))

PM25


,Samplingpoint,NatCode
0,ES/SP_02003001_9_49,02003001
1,ES/SP_09059006_9_49,09059006
2,ES/SP_28007004_9_49,28007004
3,ES/SP_28148004_9_49,28148004
4,ES/SP_02003001_9_49,02003001
5,ES/SP_09059006_9_49,09059006
6,ES/SP_28007004_9_49,28007004
7,ES/SP_28148004_9_49,28148004
8,ES/SP_02003001_9_49,02003001
9,ES/SP_09059006_9_49,09059006


PM10


,Samplingpoint,NatCode
0,ES/SP_01022001_10_47,01022001
1,ES/SP_01036004_10_47,01036004
2,ES/SP_01055001_10_47,01055001
3,ES/SP_02003001_10_49,02003001
4,ES/SP_03014009_10_46,03014009
5,ES/SP_03031002_10_46,03031002
6,ES/SP_03066003_10_46,03066003
7,ES/SP_05019002_10_49,05019002
8,ES/SP_07010001_10_47,07010001
9,ES/SP_07015001_10_49,07015001


NO2


...

,Samplingpoint,NatCode
0,ES/SP_01022001_1_38,01022001
1,ES/SP_01036004_1_38,01036004
2,ES/SP_01051001_1_38,01051001
3,ES/SP_01059008_1_38,01059008
4,ES/SP_01059009_1_38,01059009
5,ES/SP_02003001_1_38,02003001
6,ES/SP_03009006_1_38,03009006
7,ES/SP_03014006_1_38,03014006
8,ES/SP_03014008_1_38,03014008
9,ES/SP_03014009_1_38,03014009


CO


,Samplingpoint,NatCode
0,ES/SP_01022001_6_48,01022001
1,ES/SP_01055001_6_48,01055001
2,ES/SP_02003001_6_48,02003001
3,ES/SP_03009006_6_48,03009006
4,ES/SP_03014006_6_48,03014006
5,ES/SP_03014008_6_48,03014008
6,ES/SP_03014009_6_48,03014009
7,ES/SP_03031002_6_48,03031002
8,ES/SP_03065006_6_48,03065006
9,ES/SP_03065007_6_48,03065007


C6H6


,Samplingpoint,NatCode
0,ES/SP_03014006_30_59,03014006
1,ES/SP_04013004_30_59,04013004
2,ES/SP_04032005_30_I,04032005
3,ES/SP_04035002_30_I,04035002
4,ES/SP_04066004_30_I,04066004
5,ES/SP_04902001_30_I,04902001
6,ES/SP_06015001_30_59,06015001
7,ES/SP_06083001_30_59,06083001
8,ES/SP_06158001_30_59,06158001
9,ES/SP_07040002_30_59,07040002


In [5]:
# verifico que en la columna Air Quality Station Nat Code de df_stations sea el mismo formato

df_stations[["Air Quality Station Nat Code"]].head(10)

,Air Quality Station Nat Code
0,50297026
1,50297026
2,50297026
3,50297026
4,50297026
5,50297026
6,50297026
7,50297026
8,50297026
9,50297026


In [6]:
# voy a renombrar la columna Air Quality Station Nat Code de df_stations para que tenga el mismo nombre que la columna
# NatCode de los df de los contaminantes

df_stations = df_stations.rename(columns={
    "Air Quality Station Nat Code": "NatCode"
})

# aseguro que NatCode es string (por si acaso)
df_stations["NatCode"] = df_stations["NatCode"].astype(str)

print("df_stations listo conel nuevo nombre de NatCode")

df_stations listo conel nuevo nombre de NatCode


In [7]:
# verifico el cambio
df_stations[["NatCode"]].head(10)

,NatCode
0,50297026
1,50297026
2,50297026
3,50297026
4,50297026
5,50297026
6,50297026
7,50297026
8,50297026
9,50297026


In [8]:
# verifico si sale igual en los contaminantes

contaminantes["pm25"][["Samplingpoint", "NatCode"]].head(10)

,Samplingpoint,NatCode
0,ES/SP_02003001_9_49,02003001
1,ES/SP_09059006_9_49,09059006
2,ES/SP_28007004_9_49,28007004
3,ES/SP_28148004_9_49,28148004
4,ES/SP_02003001_9_49,02003001
5,ES/SP_09059006_9_49,09059006
6,ES/SP_28007004_9_49,28007004
7,ES/SP_28148004_9_49,28148004
8,ES/SP_02003001_9_49,02003001
9,ES/SP_09059006_9_49,09059006


In [ ]:
# una vez tenemos todo lo anterior hecho, voy a realizar el merge para unir cada contaminante con su estación

In [9]:
# normalizo NatCode para asegurar un formato homogéneo (string de 8 dígitos) antes del groupby de la celda siguiente

df_stations["NatCode"] = (
    df_stations["NatCode"]
    .astype(str)
    .str.strip()
    .str.zfill(8)
)

In [10]:
# creo df_stations_unique agrupando por NatCode para quedarme con una única fila por estación, ya que su información es siempre la misma

df_stations_unique = (
    df_stations
    .sort_values("NatCode")
    .groupby("NatCode")
    .first()
    .reset_index()
)

In [11]:
df_stations_unique["NatCode"].is_unique


True

In [12]:
# normalizo NatCode en todos los contaminantes para asegurar un formato homogéneo (string de 8 dígitos)

for name in contaminantes:
    contaminantes[name]["NatCode"] = (
        contaminantes[name]["NatCode"]
        .astype(str)
        .str.strip()
        .str.zfill(8)
    )


In [13]:
# filtro cada contaminante para quedarme solo con los NatCode presentes en df_stations_unique

contaminantes_filtrados = {}

for name, df in contaminantes.items():
    contaminantes_filtrados[name] = df[df["NatCode"].isin(df_stations_unique["NatCode"])]


In [14]:
# función que une un dataframe de contaminante con df_stations usando NatCode como clave única

def merge_with_stations(df_pollutant, df_stations):
    '''
    realiza un merge entre un df de un contaminante (PM25, NO2, etc.) y df_stations
    la clave de unión es NatCode, que identifica de forma única cada estación
    validate="many_to-one" garantiza que muchas mediciones se unan a una única estación
    '''
    return df_pollutant.merge(
        df_stations,
        on="NatCode",
        how="left",
        validate="many_to_one"
    )


In [15]:
# realizo el merge final uniendo cada contaminante filtrado con df_stations_unique
# guardo los dataframes resultantes en un diccionario

merged = {}

for name, df in contaminantes_filtrados.items():
    merged[name] = merge_with_stations(df, df_stations_unique)

In [16]:
# ahora voy a ver las columnas que tienen los df de contaminantes mergeados

for name, df in merged.items():
    print(f"\n{name.upper()}, columnas ({len(df.columns)}):")
    print(df.columns.tolist())



PM25, columnas (16):
['date', 'Samplingpoint', 'Value', 'NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'Air Quality Station Type', 'Altitude', 'Main Emission Sources']

PM10, columnas (16):
['date', 'Samplingpoint', 'Value', 'NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'Air Quality Station Type', 'Altitude', 'Main Emission Sources']

NO2, columnas (16):
['date', 'Samplingpoint', 'Value', 'NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'Air Quality Station Type', 'Altitude', 'Main Emission Sources']

O3, columnas (16):
['date', 'Samplingpoint', 'Value', 'NatCode', 'Year', 'Ai

In [17]:
# voy a ver las primeras filas del merge

for name, df in merged.items():
    print(name.upper())
    display(df.head())


PM25


,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2011-01-01,ES/SP_09059006_9_49,10.956522,09059006,2025,SO2,ES1443A,BURGOS 4,SP_09059006_1_38,-3.63611,42.33611,BURGOS,urban,background,929.0,"Land use, land use change and forestry"
1,2011-01-01,ES/SP_28007004_9_49,19.739130,28007004,2025,NOX as NO2,ES1890A,ALCORCÓN,SP_28007004_12_8,-3.83370,40.34190,ALCORCÓN,urban,background,693.0,NaN
2,2011-01-01,ES/SP_28148004_9_49,14.409091,28148004,2025,PM10,ES1752A,TORREJON DE ARDOZ,SP_28148004_10_49,-3.47760,40.44950,TORREJÓN DE ARDOZ,suburban,background,581.0,NaN
3,2011-01-02,ES/SP_09059006_9_49,4.500000,09059006,2025,SO2,ES1443A,BURGOS 4,SP_09059006_1_38,-3.63611,42.33611,BURGOS,urban,background,929.0,"Land use, land use change and forestry"
4,2011-01-02,ES/SP_28007004_9_49,9.291667,28007004,2025,NOX as NO2,ES1890A,ALCORCÓN,SP_28007004_12_8,-3.83370,40.34190,ALCORCÓN,urban,background,693.0,NaN


PM10


,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2013-01-01,ES/SP_01022001_10_47,6.608696,01022001,2025,SO2,ES1672A,EL CIEGO,SP_01022001_1_38,-2.61944,42.51833,ELCIEGO,suburban,traffic,480.0,NaN
1,2013-01-01,ES/SP_01036004_10_47,0.000000,01036004,2025,SO2,ES1349A,LLODIO,SP_01036004_1_38,-2.96337,43.14407,LLODIO,suburban,traffic,122.0,Transport
2,2013-01-01,ES/SP_01055001_10_47,10.782609,01055001,2025,SO2,ES1489A,VALDEREJO,SP_01055001_1_38,-3.23170,42.87520,VALDEGOVÍA,rural,background,911.0,Agriculture
3,2013-01-01,ES/SP_03014009_10_46,4.434783,03014009,2025,SO2,ES1968A,ALACANT - RABASSA,SP_03014009_1_38,-0.51389,38.35111,ALICANTE/ALACANT,suburban,industrial,20.0,Other
4,2013-01-01,ES/SP_03031002_10_46,5.260870,03031002,2025,SO2,ES1675A,BENIDORM,SP_03031002_1_38,-0.14667,38.57139,BENIDORM,suburban,background,44.0,Transport


NO2


...

,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2013-01-01,ES/SP_01022001_1_38,3.304348,01022001,2025,SO2,ES1672A,EL CIEGO,SP_01022001_1_38,-2.61944,42.51833,ELCIEGO,suburban,traffic,480.0,NaN
1,2013-01-01,ES/SP_01036004_1_38,4.000000,01036004,2025,SO2,ES1349A,LLODIO,SP_01036004_1_38,-2.96337,43.14407,LLODIO,suburban,traffic,122.0,Transport
2,2013-01-01,ES/SP_01051001_1_38,4.130435,01051001,2025,SO2,ES1544A,AGURAIN,SP_01051001_1_38,-2.39370,42.84900,SALVATIERRA O AGURAIN,suburban,background,594.0,Transport
3,2013-01-01,ES/SP_01059008_1_38,5.130435,01059008,2025,PM10,ES1502A,AVENIDA GASTEIZ,SP_01059008_10_49,-2.68070,42.85480,VITORIA-GASTEIZ,urban,traffic,517.0,Transport
4,2013-01-01,ES/SP_01059009_1_38,1.913043,01059009,2025,SO2,ES1492A,TRES MARZO,SP_01059009_1_38,-2.66779,42.85607,VITORIA-GASTEIZ,urban,traffic,518.0,Transport


CO


,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2013-01-01,ES/SP_01022001_6_48,0.191739,01022001,2025,SO2,ES1672A,EL CIEGO,SP_01022001_1_38,-2.61944,42.51833,ELCIEGO,suburban,traffic,480.0,NaN
1,2013-01-01,ES/SP_01055001_6_48,0.213043,01055001,2025,SO2,ES1489A,VALDEREJO,SP_01055001_1_38,-3.23170,42.87520,VALDEGOVÍA,rural,background,911.0,Agriculture
2,2013-01-01,ES/SP_03009006_6_48,0.139130,03009006,2025,SO2,ES1623A,ALCOI - VERGE DELS LLIRIS,SP_03009006_1_38,-0.46694,38.70639,ALCOY/ALCOI,urban,background,534.0,Transport
3,2013-01-01,ES/SP_03014006_6_48,0.147826,03014006,2025,SO2,ES1635A,ALACANT-EL PLÁ,SP_03014006_1_38,-0.47194,38.35944,ALICANTE/ALACANT,urban,traffic,45.0,Transport
4,2013-01-01,ES/SP_03014008_6_48,0.056522,03014008,2025,SO2,ES1915A,ALACANT-FLORIDA-BABEL,SP_03014008_1_38,-0.50667,38.34028,ALICANTE/ALACANT,urban,background,55.0,NaN


C6H6


,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2013-01-01,ES/SP_03014006_30_59,0.608696,03014006,2025,SO2,ES1635A,ALACANT-EL PLÁ,SP_03014006_1_38,-0.47194,38.35944,ALICANTE/ALACANT,urban,traffic,45.0,Transport
1,2013-01-01,ES/SP_04013004_30_59,1.956522,04013004,2025,PM10,ES1393A,MEDITERRÁNEO,SP_04013004_10_49,-2.44672,36.84133,ALMERÍA,urban,traffic,51.0,Transport
2,2013-01-01,ES/SP_04032005_30_I,0.420000,04032005,2025,SO2,ES1845A,PZA. DEL CASTILLO,SP_04032005_1_38,-1.89540,36.99680,CARBONERAS,urban,industrial,5.0,NaN
3,2013-01-01,ES/SP_04902001_30_I,0.550000,04902001,2025,SO2,ES1549A,EL EJIDO,SP_04902001_1_38,-2.81097,36.76972,EJIDO (EL),urban,background,97.0,Transport
4,2013-01-01,ES/SP_06015001_30_59,0.786522,06015001,2025,SO2,ES1601A,BADAJOZ,SP_06015001_1_38,-7.01139,38.88759,BADAJOZ,urban,background,390.0,Other sectors


In [18]:
for name, df in merged.items():
    print(f"\n{name.upper()}, columnas originales:")
    print([c for c in df.columns if c not in df_stations_unique.columns])

    print(f"{name.upper()}, columnas añadidas por estaciones:")
    print([c for c in df.columns if c in df_stations_unique.columns])



PM25, columnas originales:
['date', 'Samplingpoint', 'Value']
PM25, columnas añadidas por estaciones:
['NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'Air Quality Station Type', 'Altitude', 'Main Emission Sources']

PM10, columnas originales:
['date', 'Samplingpoint', 'Value']
PM10, columnas añadidas por estaciones:
['NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'Air Quality Station Type', 'Altitude', 'Main Emission Sources']

NO2, columnas originales:
['date', 'Samplingpoint', 'Value']
NO2, columnas añadidas por estaciones:
['NatCode', 'Year', 'Air Pollutant', 'Air Quality Station EoI Code', 'Air Quality Station Name', 'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality', 'Air Quality Station Area', 'A

In [19]:
for name, df in merged.items():
    print(f"{name.upper():<6} -> filas: {len(df):>7} | columnas: {len(df.columns)}")


PM25   -> filas:  292574 | columnas: 16
PM10   -> filas:  655118 | columnas: 16
NO2    -> filas: 1115131 | columnas: 16
O3     -> filas: 1181149 | columnas: 16
SO2    -> filas:  920316 | columnas: 16
CO     -> filas:  488865 | columnas: 16
C6H6   -> filas:  162114 | columnas: 16


In [20]:
# creo una lista con los dataframes de contaminantes ya mergeados con estaciones
pollutant_frames = []

for name, df in merged.items():
    tmp = df.copy()
    tmp["pollutant"] = name.upper()
    pollutant_frames.append(tmp)

# dataframe integrado final
df_pollutants_stations = pd.concat(pollutant_frames, ignore_index=True)

# guardo para el notebook 05
df_pollutants_stations.to_parquet("../data/integrated/df_pollutants_stations.parquet")

In [21]:
df_pollutants_stations

,date,Samplingpoint,Value,NatCode,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources,pollutant
0,2011-01-01,ES/SP_09059006_9_49,10.956522,09059006,2025,SO2,ES1443A,BURGOS 4,SP_09059006_1_38,-3.63611,42.33611,BURGOS,urban,background,929.0,"Land use, land use change and forestry",PM25
1,2011-01-01,ES/SP_28007004_9_49,19.739130,28007004,2025,NOX as NO2,ES1890A,ALCORCÓN,SP_28007004_12_8,-3.83370,40.34190,ALCORCÓN,urban,background,693.0,NaN,PM25
2,2011-01-01,ES/SP_28148004_9_49,14.409091,28148004,2025,PM10,ES1752A,TORREJON DE ARDOZ,SP_28148004_10_49,-3.47760,40.44950,TORREJÓN DE ARDOZ,suburban,background,581.0,NaN,PM25
3,2011-01-02,ES/SP_09059006_9_49,4.500000,09059006,2025,SO2,ES1443A,BURGOS 4,SP_09059006_1_38,-3.63611,42.33611,BURGOS,urban,background,929.0,"Land use, land use change and forestry",PM25
4,2011-01-02,ES/SP_28007004_9_49,9.291667,28007004,2025,NOX as NO2,ES1890A,ALCORCÓN,SP_28007004_12_8,-3.83370,40.34190,ALCORCÓN,urban,background,693.0,NaN,PM25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4815262,2019-12-31,ES/SP_43047001_30_59,4.262500,43047001,2025,SO2,ES1123A,CONSTANTÍ (GAUDÍ),SP_43047001_1_38,1.21770,41.15490,CONSTANTÍ,suburban,industrial,56.0,Industrial Processes,C6H6
4815263,2019-12-31,ES/SP_43103001_30_59,0.604167,43103001,2025,SO2,ES1122A,PERAFORT (PUIGDELFÍ),SP_43103001_1_38,1.23670,41.19360,PERAFORT,rural,industrial,97.0,Industrial Processes,C6H6
4815264,2019-12-31,ES/SP_46250030_30_59,2.762500,46250030,2025,SO2,ES1239A,VALÈNCIA-PISTA DE SILLA,SP_46250030_1_38,-0.37583,39.45611,VALENCIA,urban,traffic,11.0,Transport,C6H6
4815265,2019-12-31,ES/SP_47186027_30_59,2.645833,47186027,2025,PM10,ES1631A,ARCO DE LADRILLO II,SP_47186027_10_49,-4.73030,41.64560,VALLADOLID,urban,traffic,700.0,NaN,C6H6


In [ ]:

# CIERRE DEL NOTEBOOK 04

'''
en este notebook he integrado los datasets de contaminantes con los metadatos de estaciones,tras extraer y normalizar
el código nacional de estación (NatCode) en ambos lados
también he generado una versión única del dataset de estaciones y filtrado los contaminantes para asegurar
coherencia entre claves

el resultado final es df_all, un dataframe unificado que contiene todas las mediciones junto con la información
contextual de cada estación

este dataset servirá como base para el análisis exploratorio en el Notebook 05
'''